# 🛡️ AEGIS MARKETS — Clean Quant-Macro Training Pipeline

This notebook trains the **Bidirectional LSTM + Self-Attention** model using purely technical indicators and macroeconomic cross-asset signals (Gold, VIX, USD/INR, Natural Gas, S&P 500), **excluding the noisy sentiment feature**.

Once run, it will automatically generate and download the 4 essential assets for your local deployment:
1. `lstm_model_{TICKER}.h5` (Trained Model Weights)
2. `scaler_{TICKER}.pkl` (Input Feature Scaler)
3. `target_scaler_{TICKER}.pkl` (Target return Scaler)
4. `feature_config_{TICKER}.json` (Metadata configuration)

---

### Step 1: Install Libraries

In [ ]:
!pip install -q yfinance scikit-learn pandas numpy matplotlib tensorflow

### Step 2: Imports

In [ ]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

### Step 3: Configuration

**Choose the asset you want to train by changing `TICKER` below:**
- Use `^NSEI` for Nifty 50.
- Use `BZ=F` for Brent Crude Oil.

In [ ]:
# =========================================
TICKER = "^NSEI"   # "^NSEI" or "BZ=F"
# =========================================

YEARS_DATA   = 7
LOOKBACK     = 30
EPOCHS       = 100
BATCH_SIZE   = 32
TEST_SPLIT   = 0.20

CLEAN_TICKER = TICKER.replace('^', '').replace('=', '')
ASSET_NAME   = 'Nifty 50' if TICKER == '^NSEI' else 'Brent Crude Oil'
CURRENCY     = 'INR' if TICKER == '^NSEI' else 'USD'
MODEL_FILE   = f'lstm_model_{CLEAN_TICKER}.h5'
SCALER_FILE  = f'scaler_{CLEAN_TICKER}.pkl'
TARGET_SCALER_FILE = f'target_scaler_{CLEAN_TICKER}.pkl'
CONFIG_FILE  = f'feature_config_{CLEAN_TICKER}.json'

START_DATE = (datetime.now() - timedelta(days=YEARS_DATA * 365)).strftime('%Y-%m-%d')
END_DATE   = datetime.now().strftime('%Y-%m-%d')

print(f'📊 Training Asset  : {ASSET_NAME} ({TICKER})')
print(f'📅 Historical Range: {START_DATE} to {END_DATE}')
print(f'🔭 Window Lookback  : {LOOKBACK} days')

### Step 4: Download Stock Price History + Macro signals

In [ ]:
def safe_download(ticker, start, end, name):
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.empty:
            print(f"  ⚠️  {name}: Empty data")
            return None
        print(f"  ✅ {name:15s}: {len(df)} days fetched")
        return df
    except Exception as e:
        print(f"  ❌ {name} failed: {e}")
        return None

print("📥 Fetching market asset data...")
stock_raw = safe_download(TICKER, START_DATE, END_DATE, ASSET_NAME)
if stock_raw is None:
    raise ValueError(f"Failed to download data for {TICKER}")

print("📥 Fetching macroeconomic signals...")
if TICKER == '^NSEI':
    macro_tickers = {
        'USDINR=X': 'USD_INR',
        '^INDIAVIX': 'India_VIX',
        'GC=F':     'Gold',
        'BZ=F':     'Crude_Oil',
        '^GSPC':    'SP500',
    }
else:
    macro_tickers = {
        'DX-Y.NYB': 'DXY',
        'GC=F':     'Gold',
        '^GSPC':    'SP500',
        'NG=F':     'NatGas',
        '^VIX':     'VIX',
    }

macro_series_dict = {}
for t, name in macro_tickers.items():
    m_df = safe_download(t, START_DATE, END_DATE, name)
    if m_df is not None:
        close_series = m_df['Close']
        if isinstance(close_series, pd.DataFrame):
            close_series = close_series.iloc[:, 0]
        macro_series_dict[name] = close_series

### Step 5: Technical + Macro Feature Engineering

In [ ]:
df = stock_raw.copy()

# -- Technical Trend Features
df['SMA_10']   = df['Close'].rolling(10).mean()
df['SMA_20']   = df['Close'].rolling(20).mean()
df['EMA_20']   = df['Close'].ewm(span=20, adjust=False).mean()
df['EMA_50']   = df['Close'].ewm(span=50, adjust=False).mean()
df['SMA_Ratio']= df['SMA_10'] / (df['SMA_20'] + 1e-9)

# -- Momentum Features
delta = df['Close'].diff()
gain  = delta.where(delta > 0, 0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']
df['Momentum_5']  = df['Close'].pct_change(5)

# -- Volatility Features
std20 = df['Close'].rolling(20).std()
df['BB_Upper'] = df['SMA_20'] + 2 * std20
df['BB_Lower'] = df['SMA_20'] - 2 * std20
df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / (df['SMA_20'] + 1e-9)

tr1 = df['High'] - df['Low']
tr2 = (df['High'] - df['Close'].shift()).abs()
tr3 = (df['Low']  - df['Close'].shift()).abs()
true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
df['ATR'] = true_range.rolling(14).mean()
df['Rolling_Vol'] = df['Close'].pct_change().rolling(10).std()

# -- OBV Volume Feature
obv = [0]
for i in range(1, len(df)):
    if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        obv.append(obv[-1] + df['Volume'].iloc[i])
    elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        obv.append(obv[-1] - df['Volume'].iloc[i])
    else:
        obv.append(obv[-1])
df['OBV'] = np.array(obv) / 1e7

# -- Macro Signals 5d Return Features
macro_feature_names = []
for name, series in macro_series_dict.items():
    col_name = f'Macro_{name}_5d'
    pct = series.pct_change(5).rename(col_name)
    df = df.merge(pct, left_index=True, right_index=True, how='left')
    df[col_name] = df[col_name].fillna(0.0)
    macro_feature_names.append(col_name)

df.dropna(inplace=True)

FEATURE_COLS = (
    ['Close', 'Open', 'High', 'Low', 'Volume',
     'SMA_10', 'SMA_20', 'EMA_20', 'EMA_50', 'SMA_Ratio',
     'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Momentum_5',
     'BB_Upper', 'BB_Lower', 'BB_Width', 'ATR', 'Rolling_Vol',
     'OBV']
    + macro_feature_names
)

print(f"✅ Feature engineering complete! Total features: {len(FEATURE_COLS)}")
print(f"   Features: {FEATURE_COLS}")

### Step 6: Train / Test Split & Input Scaling

In [ ]:
data_matrix = df[FEATURE_COLS].values.astype(np.float32)
n_features  = data_matrix.shape[1]

split = int(len(data_matrix) * (1 - TEST_SPLIT))
train_raw = data_matrix[:split]
test_raw  = data_matrix[split:]

scaler = MinMaxScaler(feature_range=(0, 1))
train_data = scaler.fit_transform(train_raw)
test_data  = scaler.transform(test_raw)

with open(SCALER_FILE, 'wb') as f:
    pickle.dump(scaler, f)

print(f'✅ Scaler fit on Train and saved to {SCALER_FILE}')
print(f'✅ Train set size: {len(train_data)} | Test set size: {len(test_data)}')

### Step 7: Create Sequences + Scale Target (Next-Day Returns)

In [ ]:
def create_sequences(scaled_data, raw_close, lookback):
    X, y, prev_close = [], [], []
    for i in range(lookback, len(scaled_data)):
        X.append(scaled_data[i - lookback:i, :])
        ret = (raw_close[i] - raw_close[i - 1]) / (raw_close[i - 1] + 1e-9)
        y.append(ret)
        prev_close.append(raw_close[i - 1])
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32),
            np.array(prev_close, dtype=np.float32))

X_train, y_train_ret, prev_close_train = create_sequences(train_data, train_raw[:, 0], LOOKBACK)
X_test,  y_test_ret,  prev_close_test  = create_sequences(test_data,  test_raw[:, 0],  LOOKBACK)

target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train_ret.reshape(-1, 1)).flatten().astype(np.float32)
y_test  = target_scaler.transform(y_test_ret.reshape(-1, 1)).flatten().astype(np.float32)

with open(TARGET_SCALER_FILE, 'wb') as f:
    pickle.dump(target_scaler, f)

print(f'✅ Sequence datasets built successfully.')
print(f'   X_train shape: {X_train.shape}')
print(f'   Target return scaler saved to {TARGET_SCALER_FILE}')

### Step 8: Build Bidirectional LSTM + Self-Attention Model

In [ ]:
@tf.keras.utils.register_keras_serializable()
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight', shape=(input_shape[-1], 1),
            initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(
            name='attn_bias', shape=(input_shape[1], 1),
            initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

def build_model(lookback, n_features):
    inp = Input(shape=(lookback, n_features), name='price_input')
    x = Bidirectional(LSTM(128, return_sequences=True), name='bilstm_1')(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.30)(x)

    x = Bidirectional(LSTM(64, return_sequences=True), name='bilstm_2')(x)
    x = BatchNormalization()(x)
    x = SelfAttention(name='self_attention')(x)
    x = Dropout(0.20)(x)

    x = Dense(64, activation='relu', name='dense_1')(x)
    x = Dropout(0.15)(x)
    x = Dense(32, activation='relu', name='dense_2')(x)
    out = Dense(1, name='output')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='huber',
        metrics=['mae']
    )
    return model

model = build_model(LOOKBACK, n_features)
model.summary()

### Step 9: Train Model

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )
]

print("🚀 Training starting...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

### Step 10: Training Curves Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Huber Loss')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('Mean Absolute Error (MAE)')
axes[1].legend()
plt.show()

### Step 11: Validation Evaluation

In [ ]:
print('📊 Evaluating model on validation test set...')
pred_ret_scaled = model.predict(X_test, verbose=0)
pred_return = target_scaler.inverse_transform(pred_ret_scaled).flatten()

# Calculate Directional Accuracy
dir_correct = np.sign(pred_return) == np.sign(y_test_ret)
directional_acc = np.mean(dir_correct) * 100

# Calculate MAPE (Mean Absolute Percentage Error)
pred_price = prev_close_test * (1.0 + pred_return)
actual_price = prev_close_test * (1.0 + y_test_ret)
mape = np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100

print(f"\n📈 Directional Accuracy: {directional_acc:.2f}%")
print(f"📉 MAPE:                 {mape:.4f}%")

### Step 12: Save Model and Metadata Configs

In [ ]:
# Save Keras Model
model.save(MODEL_FILE)
print(f"✅ Model saved to {MODEL_FILE}")

# Save Configuration Metadata JSON
config = {
    "ticker": TICKER,
    "lookback": LOOKBACK,
    "features": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "directional_accuracy": float(round(directional_acc, 2)),
    "mape": float(round(mape, 4))
}

with open(CONFIG_FILE, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config metadata saved to {CONFIG_FILE}")

### Step 13: Automatically Download Training Assets

In [ ]:
try:
    from google.colab import files
    print("📥 Preparing assets for download...")
    files.download(MODEL_FILE)
    files.download(SCALER_FILE)
    files.download(TARGET_SCALER_FILE)
    files.download(CONFIG_FILE)
    print("🎉 Downloads initiated! Place these 4 files directly into your project's backend directory.")
except ImportError:
    print("⚠️ Running locally? Files are saved in the current directory:")
    print(f"   - {MODEL_FILE}")
    print(f"   - {SCALER_FILE}")
    print(f"   - {TARGET_SCALER_FILE}")
    print(f"   - {CONFIG_FILE}")